In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import joblib

df = pd.read_csv("../data/processed/demographic_labeled.csv")

In [ ]:
# ── Encode Gender 
df['gender_enc'] = df['gender'].str.lower().map(
    {'male': 0, 'female': 1, 'other': 0.5}
).fillna(0)



In [3]:
# ── Encode Age Group 
def encode_age(age):
    age = pd.to_numeric(age, errors='coerce')
    if pd.isna(age): return 0.5
    if age < 25:     return 0.0   # Gen Z — most S1
    elif age < 35:   return 0.25
    elif age < 50:   return 0.5
    else:            return 1.0   # Boomer — most S2

df['age_enc'] = df['age'].apply(encode_age)

In [ ]:
# ── Encode Economic Status via Membership + Spend 
membership_map = {'bronze': 0.2, 'silver': 0.5, 'gold': 0.8}
df['membership_enc'] = df['membership_type'].str.lower().map(
    membership_map
).fillna(0.5)

scaler = MinMaxScaler()
df['spend_enc'] = scaler.fit_transform(
    df[['total_spend']].fillna(df['total_spend'].median())
)
joblib.dump(scaler, "../models/spend_scaler.pkl")

['../models/spend_scaler.pkl']

In [5]:
# Encode Behavior via Items + Recency 
item_scaler = MinMaxScaler()
df['items_enc'] = item_scaler.fit_transform(
    df[['items_purchased']].fillna(5)
)
# Invert: more items = higher S1 tendency
df['items_enc'] = 1 - df['items_enc']

day_scaler = MinMaxScaler()
df['recency_enc'] = day_scaler.fit_transform(
    df[['days_since_last_purchase']].fillna(30)
)
# Invert: fewer days = more frequent = higher S1 tendency
df['recency_enc'] = 1 - df['recency_enc']

In [6]:
#  Encode Environment via City 
# Major cities = urban = more impulsive (slightly)
major_cities = [
    'new york', 'los angeles', 'chicago', 'houston', 'austin',
    'seattle', 'boston', 'miami', 'denver', 'atlanta'
]
df['environment_enc'] = df['city'].str.lower().apply(
    lambda x: 1 if any(city in str(x) for city in major_cities) else 0
)

In [7]:
#  Final feature columns 
DEMO_FEATURES = [
    'gender_enc',
    'age_enc',
    'membership_enc',
    'spend_enc',
    'items_enc',
    'recency_enc',
    'environment_enc'
]

df_features = df[DEMO_FEATURES + ['cognitive_label']].dropna()

print(f"Final demographic feature dataset: {df_features.shape}")
print(f"S1: {(df_features.cognitive_label==1).sum()} | "
      f"S2: {(df_features.cognitive_label==0).sum()}")

df_features.to_csv("../data/processed/demographic_features.csv", index=False)
print("Saved → ../data/processed/demographic_features.csv")

Final demographic feature dataset: (284, 8)
S1: 283 | S2: 1
Saved → ../data/processed/demographic_features.csv
